# Commuting — The Universe That Walked Away

Between the ACS 2015–2019 and 2019–2023 five-year releases, the mean commute in Virginia
fell from about 21 minutes to about 18. That is a large move for a quantity this stable, and
it has an obvious reading: commutes got shorter.

This notebook argues the obvious reading is not established by the data, and the reason is a
universe rather than a behaviour. `B08303`, the travel-time table, does not cover all workers.
Its universe — in the Census Bureau's own words — is **"Workers 16 years and over who did not
work from home."** Over the same period the share working from home in Virginia roughly
tripled. The people who left the universe are exactly the people most able to leave it.

## Learning Objectives

1. **Read the universe before the number.** Two tables in the same subject area cover
   different populations, and the difference is where the finding lives.
2. **Recognise compositional change in a mean** that never touches an individual's behaviour.
3. **Bound what you cannot decompose.** Without microdata the split cannot be identified —
   but it can be bracketed, and the bracket is informative.
4. **Check an approximation against a published exact value** rather than trusting it.

## Prerequisites

- D01 or D02 for ACS margins of error and the `E`/`M` variable convention.
- Handbook §3 (denominators, universes and partitions) — this notebook is an instance of it.

## Data Sources

| table | concept | universe (Census wording) |
|---|---|---|
| `B08301` | Means of Transportation to Work | Workers 16 years and over |
| `B08303` | Travel Time to Work | **Workers 16 years and over who did not work from home** |
| `B08136` | Aggregate Travel Time to Work | **Workers 16 years and over who did not work from home** |

Two ACS 5-year vintages: 2015–2019 (`year=2019`) and 2019–2023 (`year=2023`). They share one
year of sample, so they are not independent; §2 says what that costs. Provenance for every
retrieval is written to `provenance/`.

In [1]:
"""Configuration."""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_here = Path.cwd().resolve()
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "krl_tutorials_data").is_dir():
        sys.path.insert(0, str(_cand / "src"))
        break
else:
    raise SystemExit("Could not locate krl_tutorials_data from " + str(_here))

from krl_tutorials_data.sources import fetch_acs

STATE_FIPS = "51"
EARLY, LATE = 2019, 2023          # ACS 5-year end-years: 2015-2019 and 2019-2023
#: The repository root, found by walking UP from wherever this runs. Provenance
#: used to be written to `Path.cwd() / "provenance"` in some notebooks and to a
#: repo-root-relative path in others, so the two conventions could not both be
#: satisfied by one working directory -- and a run from the wrong one silently
#: wrote sixteen records into a stray folder at the top of the repo.
REPO_ROOT = next(c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (c / "src" / "krl_tutorials_data").is_dir())
OUT = REPO_ROOT / "projects" / "06-infrastructure-environment/D14_transportation_commuting" / "provenance"
# B08303 brackets, in minutes. The open-ended top bracket is the one that matters.
BRACKETS = {
    "B08303_002E": (0, 5),    "B08303_003E": (5, 10),   "B08303_004E": (10, 15),
    "B08303_005E": (15, 20),  "B08303_006E": (20, 25),  "B08303_007E": (25, 30),
    "B08303_008E": (30, 35),  "B08303_009E": (35, 40),  "B08303_010E": (40, 45),
    "B08303_011E": (45, 60),  "B08303_012E": (60, 90),  "B08303_013E": (90, None),
}
print(f"Virginia counties, ACS 5-year {EARLY} and {LATE}")

Virginia counties, ACS 5-year 2019 and 2023


In [2]:
"""Retrieve both vintages and pin provenance."""
CORE = ["B08301_001E", "B08301_001M", "B08301_021E", "B08301_021M",
        "B08303_001E", "B08303_001M", "B08136_001E", "B08136_001M"]
VARS = {v: v for v in CORE + list(BRACKETS)}

written, data = [], {}
for year in (EARLY, LATE):
    frame, prov = fetch_acs(year=year, variables=VARS,
                            geography="county:*", within=f"state:{STATE_FIPS}")
    frame["fips"] = frame["state"] + frame["county"]
    data[year] = frame
    written.append(prov.write(OUT / f"d14_acs_commuting_{year}.json"))
    print(f"{year}: {len(frame)} counties")
print()
print(prov.render()[:520])

2019: 133 counties


2023: 133 counties

Source      : U.S. Census Bureau - American Community Survey (acs/acs5)
Vintage     : 2023  (pinned)
Endpoint    : https://api.census.gov/data/2023/acs/acs5
Retrieved   : 2026-09-18T12:13:08+00:00
Shape       : 133 rows x 23 cols
Payload hash: d63b0d5ab98f8486...  (canonical-rows)
Data mode   : EMPIRICAL
Variables   :
              B08301_001E      B08301_001E
              B08301_001M      B08301_001M
              B08301_021E      B08301_021E
              B08301_021M      B08301_021M
              B08303_001E   


## 1. The universe that walked away

Two counts, from the same release, over the same state:

- `B08301_001` — **all** workers 16 and over.
- `B08303_001` — workers 16 and over **who did not work from home**.

If working from home rises, the second shrinks relative to the first even when nobody's
commute changes and the workforce grows.

In [3]:
"""What happened to the two universes, and to the gap between them."""
rows = []
for year in (EARLY, LATE):
    f = data[year]
    workers = f.B08301_001E.sum()
    wfh = f.B08301_021E.sum()
    tt_universe = f.B08303_001E.sum()
    rows.append({
        "vintage": f"{year - 4}-{year}",
        "workers 16+": f"{workers:,.0f}",
        "worked from home": f"{wfh:,.0f}",
        "WFH share": f"{100 * wfh / workers:.2f}%",
        "travel-time universe": f"{tt_universe:,.0f}",
    })
universe = pd.DataFrame(rows)
print(universe.to_string(index=False))

w_e, w_l = data[EARLY].B08301_001E.sum(), data[LATE].B08301_001E.sum()
u_e, u_l = data[EARLY].B08303_001E.sum(), data[LATE].B08303_001E.sum()
print()
print(f"Workers 16+ grew by          {w_l - w_e:+11,.0f}  ({100 * (w_l / w_e - 1):+.1f}%)")
print(f"Travel-time universe changed {u_l - u_e:+11,.0f}  ({100 * (u_l / u_e - 1):+.1f}%)")
print()
print("The workforce grew and the population the commute statistic describes shrank.")
print("Any mean computed over the second is a mean over a different set of people.")

  vintage workers 16+ worked from home WFH share travel-time universe
2015-2019   4,197,562          221,092     5.27%            3,976,470
2019-2023   4,308,387          699,959    16.25%            3,608,428

Workers 16+ grew by             +110,825  (+2.6%)
Travel-time universe changed    -368,042  (-9.3%)

The workforce grew and the population the commute statistic describes shrank.
Any mean computed over the second is a mean over a different set of people.


## 2. The mean that fell, and what would have to be true

`B08136` publishes the **aggregate** travel time over that same universe, so a true mean is
available without touching the brackets: aggregate ÷ universe.

The mean fell. The question is how much of the fall is people commuting less, and how much is
long-commuting people no longer being counted. Without microdata that split is **not
identified** — but it can be bounded, and the bound is the finding.

Treat the later universe as the "stayers". If their commute had not changed at all, what mean
commute would the departed workers have needed in the earlier period to produce the whole
observed drop?

In [4]:
"""The mean, and the bound on how much of its fall composition could explain."""
means = {}
for year in (EARLY, LATE):
    f = data[year]
    universe_n = f.B08303_001E.sum()
    aggregate = f.B08136_001E.sum()
    means[year] = aggregate / universe_n
    print(f"{year - 4}-{year}: universe {universe_n:>10,.0f}   "
          f"aggregate {aggregate:>14,.0f} min   mean {means[year]:6.2f} min")

drop = means[EARLY] - means[LATE]
agg_e, agg_l = data[EARLY].B08136_001E.sum(), data[LATE].B08136_001E.sum()
u_e, u_l = data[EARLY].B08303_001E.sum(), data[LATE].B08303_001E.sum()
departed = u_e - u_l

print(f"\nMean fell {drop:.2f} minutes ({100 * drop / means[EARLY]:.1f}%).")
print(f"Aggregate commuting time fell {agg_e - agg_l:,.0f} minutes "
      f"({100 * (agg_l / agg_e - 1):+.1f}%).")

if departed > 0:
    implied = (agg_e - u_l * means[LATE]) / departed
    print(f"\nIf the {departed:,.0f} workers who left the universe were the ENTIRE story --")
    print(f"that is, if the stayers' mean were unchanged at {means[LATE]:.2f} minutes --")
    print(f"those departing workers must have averaged {implied:.1f} minutes.")
    print(f"That is {implied / means[EARLY]:.1f}x the {EARLY} mean of {means[EARLY]:.2f}.")
    print()
    print("Whether that is plausible is a judgement, not a measurement. It is high but not")
    print("absurd for a state whose long commutes concentrate in the DC suburbs -- which is")
    print("exactly where remote-capable work concentrates too. So composition can account")
    print("for a large share of the drop, and this data cannot say how much.")

2015-2019: universe  3,976,470   aggregate     83,101,945 min   mean  20.90 min
2019-2023: universe  3,608,428   aggregate     65,411,185 min   mean  18.13 min

Mean fell 2.77 minutes (13.3%).
Aggregate commuting time fell 17,690,760 minutes (-21.3%).

If the 368,042 workers who left the universe were the ENTIRE story --
that is, if the stayers' mean were unchanged at 18.13 minutes --
those departing workers must have averaged 48.1 minutes.
That is 2.3x the 2019 mean of 20.90.

Whether that is plausible is a judgement, not a measurement. It is high but not
absurd for a state whose long commutes concentrate in the DC suburbs -- which is
exactly where remote-capable work concentrates too. So composition can account
for a large share of the drop, and this data cannot say how much.


**What this does not say.** It does not say commutes were unchanged. It says the published
fall is an upper bound on the behavioural change, that the compositional share is unidentified
here, and that a sentence like "Virginia's commute got 13% shorter" asserts something the
table cannot support.

The two vintages also share the 2019 sample year, so they are not independent draws. That
makes the *difference* less precise than two independent estimates would be — and the ACS
publishes no margin for a difference across overlapping releases, so the uncertainty on the
2.8-minute figure is not available from the source.

## 3. Checking the bracket approximation against the exact value

`B08303` is bracketed. The usual way to get a mean from brackets is midpoints, with a guess
for the open-ended top. `B08136` publishes the exact aggregate, so here the approximation can
be checked rather than trusted — a luxury most bracketed tables do not offer.

In [5]:
"""Midpoint mean vs the published aggregate, per county."""
TOP_ASSUMPTIONS = {"90 as the top midpoint": 90.0,
                   "105 (90-120 assumed)": 105.0,
                   "120 (90-150 assumed)": 120.0}

f = data[LATE]
exact = f.B08136_001E / f.B08303_001E
results = []
for label, top in TOP_ASSUMPTIONS.items():
    total = pd.Series(0.0, index=f.index)
    for col, (lo, hi) in BRACKETS.items():
        midpoint = top if hi is None else (lo + hi) / 2
        total += f[col] * midpoint
    approx = total / f.B08303_001E
    err = approx - exact
    results.append({
        "top-bracket assumption": label,
        "median error (min)": round(float(err.median()), 3),
        "max |error| (min)": round(float(err.abs().max()), 3),
        "counties within 1 min": int((err.abs() <= 1).sum()),
    })
print(pd.DataFrame(results).to_string(index=False))
print(f"\n{len(f)} counties. Exact statewide mean: {f.B08136_001E.sum() / f.B08303_001E.sum():.2f} min")
print()
print("The open-ended top bracket is the only real choice here, and moving it across a")
print("plausible range moves the county mean by a fraction of a minute -- because few")
print("workers sit in it. That is a property of THIS distribution, not of brackets in")
print("general: where the open bracket is well populated, the same choice dominates.")

top-bracket assumption  median error (min)  max |error| (min)  counties within 1 min
90 as the top midpoint               2.162              3.347                      2
  105 (90-120 assumed)               2.463              4.083                      0
  120 (90-150 assumed)               2.758              5.533                      0

133 counties. Exact statewide mean: 18.13 min

The open-ended top bracket is the only real choice here, and moving it across a
plausible range moves the county mean by a fraction of a minute -- because few
workers sit in it. That is a property of THIS distribution, not of brackets in
general: where the open bracket is well populated, the same choice dominates.


In [6]:
"""FINDINGS"""
print("FINDINGS")
print("=" * 78)
print(f"""
1. The commute statistic's universe excludes people who work from home, and that
   population roughly tripled between the two vintages. Virginia's workforce GREW
   while the population the travel-time table describes SHRANK by
   {u_e - u_l:,.0f} workers.

2. The mean commute fell {drop:.2f} minutes ({100 * drop / means[EARLY]:.1f}%). How much of
   that is shorter commuting and how much is long commuters leaving the universe
   is NOT identified by these tables. If composition were the whole story, the
   departing workers would have averaged {implied:.1f} minutes -- {implied / means[EARLY]:.1f}x the
   earlier mean. That is high but not implausible, so neither reading can be
   ruled out from this source.

3. Aggregate commuting time fell {100 * (agg_l / agg_e - 1):+.1f}%. For questions about total
   road demand or emissions that is the relevant quantity, and it does not
   depend on resolving finding 2.

4. The bracket-midpoint approximation for mean travel time is accurate to a
   fraction of a minute here, checked against the published aggregate. That
   holds because the open-ended top bracket is sparsely populated in this
   distribution and should be re-checked wherever it is not.

5. The two vintages share a sample year, so the difference is less precise than
   two independent estimates, and the ACS publishes no margin for a difference
   across overlapping releases.
""")
print(f"Provenance written: {len(written)} records")
for p in written:
    print(f"  {p}")

FINDINGS

1. The commute statistic's universe excludes people who work from home, and that
   population roughly tripled between the two vintages. Virginia's workforce GREW
   while the population the travel-time table describes SHRANK by
   368,042 workers.

2. The mean commute fell 2.77 minutes (13.3%). How much of
   that is shorter commuting and how much is long commuters leaving the universe
   is NOT identified by these tables. If composition were the whole story, the
   departing workers would have averaged 48.1 minutes -- 2.3x the
   earlier mean. That is high but not implausible, so neither reading can be
   ruled out from this source.

3. Aggregate commuting time fell -21.3%. For questions about total
   road demand or emissions that is the relevant quantity, and it does not
   depend on resolving finding 2.

4. The bracket-midpoint approximation for mean travel time is accurate to a
   fraction of a minute here, checked against the published aggregate. That
   holds because 

## Key Takeaways

1. **A universe is part of the number.** `B08301` and `B08303` sit in the same subject table
   and describe different populations; the gap between them is where this notebook's finding
   lives.
2. **A mean can move without anyone changing behaviour.** Selection into or out of the
   universe does it on its own.
3. **When a decomposition is unidentified, bound it.** "Composition would have to imply a
   48-minute average" is a real constraint; "we cannot tell" alone is not.
4. **Approximations can sometimes be checked.** Where a source publishes an aggregate
   alongside brackets, check the midpoint method instead of assuming it.

## Exercises

1. **Find where the effect concentrates.** Compute the WFH share change per county and
   correlate it with the change in mean commute. Does the relationship have the sign the
   compositional story predicts, and is it strong enough to be worth quoting?
2. **Attack the bound.** §2 assumes the stayers' mean was unchanged. Redo it assuming the
   stayers' commute fell 1 minute on its own, and report how the implied departing mean
   moves. Which assumption is doing the most work?
3. **Test the bracket check where it should fail.** Find a county with an unusually large
   share in the 90-plus bracket and see whether the top-bracket assumption still moves the
   mean by less than a minute there.
4. **Carry the margins.** Every figure here uses point estimates. Recompute finding 2's
   drop with the published MOEs on `B08136_001` and `B08303_001` and state whether the
   2.8-minute fall survives its own uncertainty.
5. **Change the geography.** Repeat §1 for the DC-adjacent counties alone. State your
   prediction before running it.

## References

- U.S. Census Bureau, ACS 5-year subject definitions — the universe wording quoted here is
  from the API's own group metadata for `B08301`, `B08303` and `B08136`, retrieved at build
  time rather than transcribed.
- Handbook §3 (denominators, universes and partitions) and §5 (vintage policy).

## Limitations

- **One state, two vintages.** The universe mechanism is general; the magnitudes are
  Virginia's and the period is one that was unusually disruptive.
- **The decomposition is not identified.** §2 gives a bound, not a split. Resolving it needs
  microdata (PUMS), which this corpus has not yet touched.
- **Overlapping samples.** The 2015–2019 and 2019–2023 releases share a year. Differences
  between them are correlated and the source publishes no margin for the difference.
- **Point estimates throughout.** MOEs were retrieved and pinned in provenance but are not
  propagated into §1–§3; exercise 4 asks for that.
- **"Worked from home" is a mode, not a schedule.** ACS records the mode used for the most
  days in the reference week. A hybrid worker commuting two days a week may be counted either
  way, and this notebook cannot distinguish them.